# Partitioned Data Lake Architecture: Designing Directory Layouts

**Module**: 08_data_sources_and_formats  
**Notebook**: 03_partitioned_data_lake.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Designing partition directories (`year=YYYY/month=MM/`), directory partitioning strategies, and partition pruning.
- **Why does this topic exist?**: Directory partitioning enables queries to skip 90%+ of files during scanning.
- **What problem does it solve?**: Creating too many partitions (over-partitioning) or partitioning on high-cardinality keys.
- **Where is it used in real-world Data Engineering?**: Enterprise data lake storage layout design.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Parquet/ORC, partitioning strategies |
| **Required depth** | Working knowledge: can design and query a partitioned lake layout |
| **Data Engineer** | Design partition columns that match access patterns and avoid small-file explosions. |
| **Senior Data Engineer** | Diagnose partition pruning failures and manage partition metadata. |
| **Spark specialist** | Reason about partition discovery, overwrite semantics, and table-format alternatives. |
| **Next topic** | `08_data_sources_and_formats/04_schema_evolution.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A partitioned lake stores files under directories such as `date=2026-01-01/`, letting readers skip
irrelevant data. The design is a contract with every downstream query: choose the wrong column and you create
millions of tiny partitions; choose the right one and scans shrink by orders of magnitude. Partitioning is not the
same as bucketing or clustering, and it is not a substitute for file compaction.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Partition column** | Directory key | Values become directory names and are not stored in the files by default |
| **Partition pruning** | Skip directories | Requires a filter on the partition column |
| **Partition discovery** | Infer partitions from paths | Can be expensive with many directories |
| **Dynamic overwrite** | Replace only touched partitions | `spark.sql.sources.partitionOverwriteMode=dynamic` |
| **Small files** | Many tiny files per partition | Caused by high-cardinality partitioning or frequent appends |
| **Metastore sync** | Register partitions | `MSCK REPAIR TABLE` or automatic discovery |

### How it works, step by step

1. The writer creates one directory per partition value.
2. Files inside a partition contain only rows for that value.
3. A reader parses the path, discovers partition columns, and builds a directory list.
4. Filters on partition columns prune directories before opening files.
5. The remaining files are read with column pruning and predicate pushdown.
6. New partitions must be discovered or registered before they are queryable through the metastore.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Data Lake Partition Directory Hierarchy: `warehouse/orders/year=2025/month=09/data.parquet`

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Data_Lake_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
out_lake = "/opt/spark-data/warehouse/lake_orders"

# Add year and month partition columns
lake_df = orders.withColumn("year", F.year("order_ts")).withColumn("month", F.month("order_ts"))
lake_df.write.mode("overwrite").partitionBy("year", "month").parquet(out_lake)
print("Data Lake partitioned layout created successfully!")

---

## Experiment: Partition Pruning Verification

- **1. What are we testing?**: Filter on year=2025 and month=9 and verify that only 1 directory is scanned.
- **2. What do we expect?**: PartitionFilters skips all other month directories.

In [ ]:
read_lake = spark.read.parquet(out_lake)
read_lake.filter("year = 2025 AND month = 9").explain("formatted")

**3. What happened?**: PartitionFilters pushed to FileScan; irrelevant directories were ignored.  
**4. Why did it happen?**: Spark pruned unneeded directories before listing files.  
**5. Practical Engineering Lesson**: Partition by query filter columns to accelerate analytical scans.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Date-partitioned event lake

- **Industry**: Advertising
- **Scenario**: Billions of events are queried mostly by date and campaign.
- **Problem**: A flat layout forces every query to scan all history.
- **Why this concept is relevant**: Partitioning by date prunes most files for time-bounded queries; campaign can be a secondary partition or a clustering key.
- **Implementation idea**: Measure scan bytes for representative queries and choose the layout with the best pruning without excessive directories.
- **Expected result**: Queries scan only the relevant dates.
- **Alternative**: Use a table format with hidden partitioning.
- **Trade-offs**: Hidden partitioning avoids path conventions but adds a dependency and migration work.

### Use Case 2: Overwriting one day without deleting everything

- **Industry**: Logistics
- **Scenario**: A backfill must replace one day's partition while preserving the rest.
- **Problem**: A static overwrite mode can delete the entire table output.
- **Why this concept is relevant**: Dynamic partition overwrite replaces only the partitions present in the new data.
- **Implementation idea**: Set the overwrite mode, write the backfill partition, and verify that other dates remain.
- **Expected result**: A safe backfill with no collateral deletion.
- **Alternative**: Write to a staging table and swap partitions.
- **Trade-offs**: A staging table is safer for complex backfills but adds orchestration.

---

## Business Value

- Large reductions in scanned data and cost.
- Safe backfills with partition-scoped overwrites.
- Predictable query performance for time-series data.
- Clear layout ownership across ingestion and analytics teams.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Partition on low-to-moderate cardinality columns used in filters.
- Avoid partitioning by user ID or exact timestamps.
- Compact small files regularly.
- Verify partition pruning in the plan.
- Use dynamic overwrite for backfills and test on a copy first.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Prove partition pruning

**Steps**

1. Write a dataset partitioned by date.
2. Query a single date and inspect `PartitionFilters` in the scan.
3. Compare scanned files and bytes with a query that filters on a non-partition column.

**What to observe**: Only the matching date directories are read when the filter uses the partition column.

**Healthy result**: The plan lists the expected partition filter.

**Problematic result**: The filter is applied after a cast or a view, so all partitions are read.

**Engineering takeaway**: Pruning depends on the filter reaching the partition column.

---

## Trade-offs and Alternatives

### Advantages

- Cheap pruning
- Simple directory model
- Portable across engines

### Limitations

- High cardinality creates small files
- Layout changes are expensive
- Metadata discovery can be slow

### When to use it

- Date-filtered event data
- Daily batch outputs
- Backfills

### When not to use it

- Unique-key partitioning
- Over-partitioning
- Assuming pruning without checking the plan

### Alternatives inside Spark

- partitionBy
- Partition discovery
- Dynamic overwrite

### Alternatives outside Spark

- Iceberg/Delta/Hudi
- Warehouse partitioning
- Clustering

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Inspect Directory Tree
**Problem Statement**: List generated directories under out_lake.  
**Input / Context**: out_lake

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Filter Single Month
**Problem Statement**: Count orders in month 11.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Max Records Per File
**Problem Statement**: Write with maxRecordsPerFile setting.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The backfill that deleted history

**Scenario**: A backfill overwrites one day but the table loses all previous dates.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check `spark.sql.sources.partitionOverwriteMode`.
2. Confirm whether the write used `overwrite` on the table root.
3. Restore from a snapshot or backup and switch to dynamic overwrite.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Inspect Directory Tree):
```python
print([d for d in os.listdir(out_lake) if d.startswith('year')])
```

#### Solution 2 (Filter Single Month):
```python
print('Count:', read_lake.filter('month = 11').count())
```

#### Solution 3 (Max Records Per File):
```python
lake_df.write.mode('overwrite').option('maxRecordsPerFile', 5000).parquet('/opt/spark-data/warehouse/lake_chunked')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Partition Pruning**: Skips entire directories at the storage layer.
- **Cardinality Rule**: Use low/medium cardinality columns (year, month, region).

### Essential Technical Terms
- `partitionBy`: Writes data into nested subdirectories.
- `PartitionFilters`: Physical operator pruning directories.

### Common Engineering Mistakes
- **Partitioning by timestamp**: Creates millions of empty directories, destroying file system performance.

### Interview Questions on This Topic
- **Q: What is partition pruning?**
  - *A*: Spark reading only matching partition directory folders, ignoring all other folders entirely.


### More Interview Questions

- **Q: What is partition pruning?**
  - *A*: Skipping directories/files whose partition values cannot satisfy the query filter, which reduces the data read.
- **Q: Why is partitioning by user ID a bad idea?**
  - *A*: It creates a huge number of directories and small files, increasing metadata overhead and often hurting scan performance.

### What to Study Next
Open 04_schema_evolution.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')